In [ ]:
!pip install pandas-profiling
!pip uninstall -y pandas-profiling numba visions
!pip install ydata-profiling

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.6/262.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.4/102.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.8/309.8 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 24.9 MB/s eta 0:00:00
  Created wheel for htmlmin: filename=htmlmin-0.1.12-py3-none-any.whl size=27081 sha256=53df9869e247ba06436126f3cdda623f4e863e2ff78346cd5514acecd3ae2852
  Stored in directory: /root/.cache/pip/wheels/5f/d4/d7/4189b07b5902ee9f3ce0dbb14909fbe8037c39d6c63ffd49c9
Successfully built htmlmin
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.3
    Uninstalling MarkupSafe-3.0.3:
      Successfully uninstalled MarkupSafe-3.0.3
  Attempting uninstall:

In [ ]:
import kagglehub
import pickle
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn  import svm

In [ ]:
# Download latest version
path = kagglehub.dataset_download("yasserh/titanic-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'titanic-dataset' dataset.
Path to dataset files: /kaggle/input/titanic-dataset


In [ ]:
df=pd.read_csv("/kaggle/input/titanic-dataset/Titanic-Dataset.csv")

In [ ]:
from ydata_profiling import ProfileReport
prof=ProfileReport(df)
prof.to_file(output_file='output.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 12/12 [00:01<00:00,  9.23it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
df.drop(columns=['PassengerId','Cabin','Name','Ticket'],axis=1,inplace=True)

In [ ]:
x1=df.drop(columns=['Survived'])
y1=df['Survived']

In [ ]:


x_train2,x_test2,y_train2,y_test2=train_test_split(x1,y1,test_size=0.2,random_state=42)

In [ ]:

def create_age_group(X):
    X = X.copy()
    labels = ['Child','Teenager','Adult','Senior','Old']

    X['age_group'] = pd.cut(
        X['Age'],
        bins=[0,12,18,35,60,80],
        labels=labels
    )

    return X

In [ ]:
age_transformer = FunctionTransformer(create_age_group)

In [ ]:
preprocess=ColumnTransformer([
    ('num',Pipeline([
         ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]),['Age', 'Fare','Pclass','SibSp','Parch']),

    ('cat',Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]),['Sex', 'Embarked','age_group'])
])

In [ ]:
pipe=make_pipeline(age_transformer,preprocess,svm.SVC())

In [ ]:
pipe.fit(x_train2,y_train2)

Pipeline(steps=[('functiontransformer',
                 FunctionTransformer(func=<function create_age_group at 0x7ad34d4a7100>)),
                ('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare', 'Pclass',
                                                   'SibSp', 'Parch']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'Embarked',
                                                   'age_group'])])),
                ('svc', SVC())])

In [ ]:
ypred3=pipe.predict(x_test2)

In [ ]:
print(accuracy_score(y_test2,ypred3))

0.8212290502793296


In [ ]:

# Save the trained SVM model to a file
filename = 'svm_model.pkl'
pickle.dump(pipe, open(filename, 'wb'))

print(f"SVM model successfully saved to {filename}")

SVM model successfully saved to svm_model.pkl
